<a href="https://colab.research.google.com/github/sujeet-banerjee/colab-genai-scratch/blob/sujeet-banerjee-patch-1/RAG_load_raw_docs_videos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Load a doc from the Google Drive

# Load docs from various sources


In [43]:
! pip install --upgrade pip
! pip install openai
! pip install pypdf
!pip install langchain_community

!pip install langchain
!pip install langgraph
!pip install langchain-core
!pip install langchain-openai
!pip install --upgrade langchain

Load Open AI Client

In [44]:
from openai import OpenAI

# For secrets
from google.colab import userdata

open_api_key = userdata.get('open_api_key')
c_key = userdata.get('c_key')
hf_sujeet = userdata.get('hf_sujeet')

client = OpenAI(
  api_key= open_api_key
)

Load PDF/file from Google Drive!

In [45]:
from google.colab import drive
drive.mount('/content/drive')

!ls /content/drive/MyDrive

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
'A B Test.gform'
'A B Test (Responses).gsheet'
 APAL_Study
 Blue_laptop_backup
 Code
'Colab Notebooks'
'Customer Feedback.gform'
'Docker Swarm with Mixed OS (Windows and Ubuntu).gdoc'
 Dot_product.drawio
 Emulator_No_Begin.png
 Emulator_No_singnin.png
 GenAI_Posts
'Github Token No Expiry.gdoc'
'give example python code for PEFT based on adapte...'
'Hands-On Generative AI Techniques: A Guided Journey.gdoc'
'how meta AI seach is going to dent Google?.gdoc'
 icons
'India Employment Agreement_SIGNED_AND_ACCEPTED.pdf'
'India PIIA.pdf'
 Jobs
'Leetcode for RateLimit UniqueNameGen and GroupMembership.gdoc'
'Open-source agentic AI platforms.'$'\n'' (1).gdoc'
'Open-source agentic AI platforms.'$'\n''.gdoc'
'Permissions for '\''suzlab.pem'\'' are too open. how to set pem perms using powershell?.gdoc'
'Prep Doc for Coding & Design.pdf'
'prove that 2^n > n!.gdoc'
'reverse

# Spit or Do chunking

In [46]:
from langchain.text_splitter import RecursiveCharacterTextSplitter, CharacterTextSplitter, TokenTextSplitter

#Test chunking
chunk_size =120
chunk_overlap = 24

text_splitter = TokenTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap
)

r_splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
)
c_splitter = CharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
    separator = ' '
)





In [47]:
texts = text_splitter.split_documents(pages)
print(len(texts))
texts[0].page_content[0:20]

905


'Human Resource Analy'

In [48]:
r_splitter.split_text(texts[0].page_content)

['Human Resource Analytics \n(Why are our employees leaving?) \n \nBackground',
 'Background \nHari Prasad, the senior manager (HR) of Chaurasia and Company was going through the employee',
 'records of his organization. The most startling fact that caught his eye were the names Amar Akbar',
 'and Anthony. All the three were top performing and experienced employees with whom he had',
 'recently interacted in an annual meeting. But his anxiety became worse as he noticed that this was',
 'a pattern- some of the experienced and valuable employees had recently left the company']

In [49]:
text_splitter.split_text(texts[0].page_content)

['Human Resource Analytics \n(Why are our employees leaving?) \n \nBackground \nHari Prasad, the senior manager (HR) of Chaurasia and Company was going through the employee \nrecords of his organization. The most startling fact that caught his eye were the names Amar Akbar \nand Anthony. All the three were top performing and experienced employees with whom he had \nrecently interacted in an annual meeting. But his anxiety became worse as he noticed that this was \na pattern- some of the experienced and valuable employees had recently left the company']

In [50]:
c_splitter.split_text(texts[0].page_content)

['Human Resource Analytics \n(Why are our employees leaving?) \n \nBackground \nHari Prasad, the senior manager (HR) of',
 'senior manager (HR) of Chaurasia and Company was going through the employee \nrecords of his organization. The most',
 'organization. The most startling fact that caught his eye were the names Amar Akbar \nand Anthony. All the three were top',
 'All the three were top performing and experienced employees with whom he had \nrecently interacted in an annual meeting.',
 'in an annual meeting. But his anxiety became worse as he noticed that this was \na pattern- some of the experienced and',
 'of the experienced and valuable employees had recently left the company']

### Context aware splitting

Context aware splitting
Chunking aims to keep text with common context together.

A text splitting often uses sentences or other delimiters to keep related text together but many documents (such as Markdown) have structure (headers) that can be explicitly used in splitting.

We can use MarkdownHeaderTextSplitter to preserve header metadata in our chunks, as show below.

In [51]:
# prompt: MarkdownHeaderTextSplitter

from langchain.text_splitter import MarkdownHeaderTextSplitter

markdown_document = """
# Title

## Header 1
This is some text under Header 1.

### Sub-header 1.1
This is some text under Sub-header 1.1.

## Header 2
This is some text under Header 2.
"""

headers_to_split_on = [
    ("#", "TitleSuz"),
    ("##", "HeaderSuz"),
    ("###", "Sub-header-suz"),
]

markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on
)

md_header_splits = markdown_splitter.split_text(markdown_document)

for split in md_header_splits:
    print(split.metadata)
    print("---")

{'TitleSuz': 'Title', 'HeaderSuz': 'Header 1'}
---
{'TitleSuz': 'Title', 'HeaderSuz': 'Header 1', 'Sub-header-suz': 'Sub-header 1.1'}
---
{'TitleSuz': 'Title', 'HeaderSuz': 'Header 2'}
---


# Embeddings and Vector-DB storage

#### RecursiveCharacterTextSplitter

In [52]:
r_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=150,
    separators=["\n\n", "\n", "(?<=\. )", " ", ""]
)

#Cunked docs
docs = r_splitter.split_documents(pages);
print('Pages len: ', len(pages))
print('Docs len: ', len(docs))

print(docs[0:5])


Pages len:  220
Docs len:  355
[Document(metadata={'producer': 'Adobe PDF Library 11.0', 'creator': 'Acrobat PDFMaker 11 for Word', 'creationdate': '2024-04-14T17:42:26+05:30', 'author': 'IIMT', 'comments': '', 'company': '', 'keywords': '', 'moddate': '2024-04-14T17:42:28+05:30', 'sourcemodified': 'D:20240414085852', 'subject': '', 'title': '', 'source': '/content/drive/MyDrive/APAL_Study/Case_HRA_IIMC.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}, page_content='Human Resource Analytics \n(Why are our employees leaving?) \n \nBackground \nHari Prasad, the senior manager (HR) of Chaurasia and Company was going through the employee \nrecords of his organization. The most startling fact that caught his eye were the names Amar Akbar \nand Anthony. All the three were top performing and experienced employees with whom he had \nrecently interacted in an annual meeting. But his anxiety became worse as he noticed that this was \na pattern- some of the experienced and valuable employees

#### Create Embeddings using OpenAI API

In [53]:
from langchain.embeddings import OpenAIEmbeddings as oaE
# Alternative
from langchain.embeddings import HuggingFaceEmbeddings as hfE
import numpy as np

#print(open_api_key)

In [54]:
#Using Open API Embeddings
'''
dot of vec1 and vec2 0.9630313341962432
dot of vec1 and vec3 0.7701484645728846
'''
#

s1 = "i like dogs"
s2 = "i like canines"
s3 = "the weather is ugly outside"

vec1 = oaE(openai_api_key=open_api_key).embed_query(text=s1)
vec2 = oaE(openai_api_key=open_api_key).embed_query(text=s2)
vec3 = oaE(openai_api_key=open_api_key).embed_query(text=s3)

print(vec1)
print(vec2)
print(vec3)
print(np.linalg.norm(vec1))
print(np.linalg.norm(vec2))
print(np.linalg.norm(vec3))
print("dot of vec1 and vec2", np.dot(vec1, vec2))
print("dot of vec1 and vec3", np.dot(vec1, vec3))

print("Vector length OpenAI: ", len(vec1), "")

[-0.027551146146196668, -0.005382069836976813, -0.02582130760099973, -0.03305632611697975, -0.027349121791237472, 0.022639416094150544, -0.010435849398653005, -0.008125188737472597, 0.0025079497065223677, -0.019646922070829638, 0.000673547850878873, 0.02924310407710086, -0.00539785296060417, 0.0007114275338490437, 0.0002138622943082356, 0.0140786105368597, 0.029874432747485417, -0.001194787906827953, 0.004210956382759252, -0.0038700394688583597, -0.011692191441061522, 0.006875160130779538, 0.012992726602155665, -0.04724857570274996, -0.0022980333240882078, 0.004706548979229199, 0.016932211768408267, -0.00013178968182733469, -0.02588444009550916, -0.016351390583747363, 0.02669254124063623, 0.0029972288672768573, -0.01551803755840361, -0.024760676477834954, 0.006035494135381614, -0.014949843245173624, 0.009785581352444648, -0.011515419897641582, -0.004349848708870307, -0.01087777872286414, -0.017058478620072263, 0.011180816186625503, 0.006938292625288966, -0.02204596803805873, -0.0025758

In [55]:
## Using HuggingFace Embeddings
# Result
'''
dot of vec1 and vec2 0.8981182842559048
dot of vec1 and vec3 0.03641026493961007
'''
#

s1 = "i like dogs"
s2 = "i like canines"
s3 = "the weather is ugly outside"

vec1 = hfE().embed_query(s1)
vec2 = hfE().embed_query(s2)
vec3 = hfE().embed_query(s3)

print(vec1)
print(vec2)
print(vec3)
print(np.linalg.norm(vec1))
print(np.linalg.norm(vec2))
print(np.linalg.norm(vec3))
print("dot of vec1 and vec2", ( np.dot(vec1, vec2)/(np.linalg.norm(vec1) * np.linalg.norm(vec2)) ))
print("dot of vec1 and vec3", ( np.dot(vec1, vec3)/(np.linalg.norm(vec1) * np.linalg.norm(vec3)) ) )

print("Vector length HuggingFace: ", len(vec1))

/tmp/ipython-input-3719861435.py:13: LangChainDeprecationWarning: Default values for HuggingFaceEmbeddings.model_name were deprecated in LangChain 0.2.16 and will be removed in 0.4.0. Explicitly pass a model_name to the HuggingFaceEmbeddings constructor instead.
  vec1 = hfE().embed_query(s1)
/tmp/ipython-input-3719861435.py:14: LangChainDeprecationWarning: Default values for HuggingFaceEmbeddings.model_name were deprecated in LangChain 0.2.16 and will be removed in 0.4.0. Explicitly pass a model_name to the HuggingFaceEmbeddings constructor instead.
  vec2 = hfE().embed_query(s2)
/tmp/ipython-input-3719861435.py:15: LangChainDeprecationWarning: Default values for HuggingFaceEmbeddings.model_name were deprecated in LangChain 0.2.16 and will be removed in 0.4.0. Explicitly pass a model_name to the HuggingFaceEmbeddings constructor instead.
  vec3 = hfE().embed_query(s3)


[-0.013610822148621082, 0.11503763496875763, -0.027678335085511208, -0.041481710970401764, 0.04854816943407059, 0.03823169320821762, -0.045007575303316116, -0.012027067132294178, -0.00249747047200799, -0.017130006104707718, -0.08121422678232193, 0.017644111067056656, -0.05390755832195282, -0.013921351172029972, -0.017844628542661667, -0.011630043387413025, 0.03812805190682411, 0.08376377075910568, 0.016784565523266792, -0.010697281919419765, 3.216173354303464e-05, 0.030750378966331482, -0.05290459841489792, -0.03610792011022568, 0.023436011746525764, 0.03191141411662102, -0.025538088753819466, -0.0515754334628582, 0.035590071231126785, 0.03687377646565437, -0.05278579145669937, -0.029876425862312317, -0.012499134987592697, -0.02516518533229828, 1.4985516827437095e-06, 0.01840418577194214, -0.03838668018579483, 0.020760133862495422, 0.060190360993146896, -0.0023674792610108852, -0.03244411200284958, -0.06694810837507248, -0.038730520755052567, -0.05493103712797165, -0.03285567834973335,

#### Store in Chroma DB (Vector Store)

In [56]:
! pip install chromadb
! pip install tiktoken

##### Using HuggingFace for Apriel (Service Now)

@Read:
https://huggingface.co/ServiceNow-AI/Apriel-5B-Base

###### To load chromaDB from a handful of texts:

texts = [
    """The Amanita phalloides has a large and imposing epigeous (aboveground) fruiting body (basidiocarp).""",

    """A mushroom with a large fruiting body is the Amanita phalloides. Some varieties are all-white.""",
    
    """A. phalloides, a.k.a Death Cap, is one of the most poisonous of all known mushrooms.""",
]

smalldb = Chroma.from_texts(texts, embedding=embedding)



###### To load from a disk / G-drive
vectordb = Chroma(persist_directory=persist_directory, embedding_function=embedding)

In [69]:
from langchain.vectorstores import Chroma

persist_directory = '/content/drive/MyDrive/vectordb/chroma_db'
# !rm -rf persist_directory  # remove old database files if any
# Change persist_directory to a location within the Colab environment's writable file system

# Does not work!
#embedding=hfE(model_name="ServiceNow/msrc-attendee-apriel-small-v2", trust_remote_code=True),

'''
# First Time
vectordb = Chroma.from_documents(
    documents=docs,
    embedding=hfE(),
    persist_directory=persist_directory
)
'''
# Next time onwards
vectordb = Chroma(
    persist_directory=persist_directory,
    embedding_function=hfE())

print(vectordb._collection.count())

/tmp/ipython-input-695283669.py:21: LangChainDeprecationWarning: Default values for HuggingFaceEmbeddings.model_name were deprecated in LangChain 0.2.16 and will be removed in 0.4.0. Explicitly pass a model_name to the HuggingFaceEmbeddings constructor instead.
  embedding_function=hfE())


355


/tmp/ipython-input-695283669.py:19: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vectordb = Chroma(


##### Persist DB And Query

In [58]:
vectordb.persist()


In [70]:
question = "is there an email i can ask for help"
qe = vectordb.similarity_search(question,k=3)
print("Searched docs: ", len(qe))
print("\n\n-----\n".join([x.page_content for x in qe]))


Searched docs:  3
Components of a Good Prompt
• Persona
• Task
• Context
• Format
Help me craft an empathetic email response. I am
a customer service representative, and I need to
create a response to a customer complaint. The
customer ordered a pair of headphones that
arrived damaged. They’ve already contacted us
via email and provided pictures of the damage.
I’ve offered a replacement, but they’re requesting
an expedited shipping option that isn’t typically
included with their order. Include a paragraph
that acknowledges their frustration and three
bullet points with potential resolutions. 
Source: Gemini for Google Workspace: Prompting Guide 101

-----
Components of a Good Prompt
• Persona
• Task
• Context
• Format
Help me craft an empathetic email response. I am 
a customer service representative, and I need to 
create a response to a customer complaint. The 
customer ordered a pair of headphones that 
arrived damaged. They’ve already contacted us 
via email and provided pictures o

In [60]:
question = "what did they say about the supply chain in the chapter Impacts of technology disruption?"
qe = vectordb.similarity_search(question,k=3)
print("Searched docs: ", len(qe))
print("\n\n-----\n".join([x.page_content for x in qe]))

Searched docs:  3
Impacts of technology 
disruption
From a supply chain to a DSN
T
HE function of any supply chain centers on 
the movement of materials, finished goods, 
capital, and other assets from place to place, 
as well as the production of finished goods. At their 
core, however, supply chains consist of many trans-
actions: the exchange of time, money, information, 
or physical materials for some other unit of value. 
Dramatic technological and digital developments, 
such as greater computing power and lower overall 
costs, have impacted the traditional supply chain in 
several key ways, including a reduction in transac-
tion costs and increase in innovation related to the 
production process itself.
Reducing transaction costs
The increase in power and efficiency of technologies 
has manifested itself in greatly reduced transaction 
costs for business operations both internally and ex-
ternally.8 No longer does it have to be prohibitively 
expensive or time intensive to gain i

##### Problems with the above vector search
1. There is no diversification of the responses. Sometimes there are a repeat of the chunks, adding no value to the final prompting to the LLM
2. Often the most relevant chunk might be at teh bottom (or out) of the reported list, esp with specified filters.

Solution:
1. Using diversification of the reported docs - *MMR*
2. filter the obvious records (say, chapter name etc), by specifying the query-filters passed on to the db search

In [61]:
question = "is there an email i can ask for help"
qe = vectordb.max_marginal_relevance_search(question, k=3, fetch_k=10)
print("Searched docs: ", len(qe))
print("\n\n-----\n".join([x.page_content for x in qe]))

Searched docs:  3
Components of a Good Prompt
• Persona
• Task
• Context
• Format
Help me craft an empathetic email response. I am
a customer service representative, and I need to
create a response to a customer complaint. The
customer ordered a pair of headphones that
arrived damaged. They’ve already contacted us
via email and provided pictures of the damage.
I’ve offered a replacement, but they’re requesting
an expedited shipping option that isn’t typically
included with their order. Include a paragraph
that acknowledges their frustration and three
bullet points with potential resolutions. 
Source: Gemini for Google Workspace: Prompting Guide 101

-----
CONTACTS
Adam Mussomeli
Principal
Supply Chain and Manufacturing Operations
Deloitte Consulting LLP
+1 203 253 5101
amussomeli@deloitte.com
Doug Gish
Principal
Supply Chain and Manufacturing Operations
Deloitte Consulting LLP
+1 816 802 7270
dgish@deloitte.com
Stephen Laaper
Principal
Supply Chain and Manufacturing Operations
Deloitte

# Retrieval

In [62]:
!pip install lark

#### MMR - Max Marginal Relevance
We saw the example above using
> vectordb.max_marginal_relevance_search(question, k=3, fetch_k=10)

MMR brings more diversified results as opposed to the very high ranked similarity matches.



#### Specify DB filter

In [63]:
question = "what did they say about the supply chain in the chapter Impacts of technology disruption?"
qe = vectordb.similarity_search(
    question,
    k=3,
    filter={"source":"/content/drive/MyDrive/APAL_Study/DUP_Digital-supply-network.pdf"}
    )
print("Searched docs: ", len(qe))
print("\n\n-----\n".join([x.page_content for x in qe]))

Searched docs:  3
Impacts of technology 
disruption
From a supply chain to a DSN
T
HE function of any supply chain centers on 
the movement of materials, finished goods, 
capital, and other assets from place to place, 
as well as the production of finished goods. At their 
core, however, supply chains consist of many trans-
actions: the exchange of time, money, information, 
or physical materials for some other unit of value. 
Dramatic technological and digital developments, 
such as greater computing power and lower overall 
costs, have impacted the traditional supply chain in 
several key ways, including a reduction in transac-
tion costs and increase in innovation related to the 
production process itself.
Reducing transaction costs
The increase in power and efficiency of technologies 
has manifested itself in greatly reduced transaction 
costs for business operations both internally and ex-
ternally.8 No longer does it have to be prohibitively 
expensive or time intensive to gain i

#### Self Query Retriever



Addressing Specificity: working with metadata using self-query retriever
But we have an interesting challenge: we often want to infer the metadata from the query itself.

To address this, we can use SelfQueryRetriever, which uses an LLM to extract:

The query string to use for vector search
A metadata filter to pass in as well
Most vector databases support metadata filters, so this doesn't require any new databases or indexes.

In [64]:
#Check the vector DB count
print(vectordb._collection.count())

355


In [65]:
from langchain.llms import OpenAI
from langchain.retrievers.self_query.base import SelfQueryRetriever
from langchain.chains.query_constructor.base import AttributeInfo

metadata_field_info = [
    AttributeInfo(
        name="source",
        description="The document the chunk is from, for example 'DUP_Digital-supply-network.pdf'",
        type="string",
    ),
    AttributeInfo(
        name="page",
        description="The page or paragraph from the document",
        type="integer",
    ),
    AttributeInfo(
        name="chapter",
        description="The chapter from the document, containing multiple pages",
        type="integer",
    ),
]

document_content_description = "Documents"
llm = OpenAI(
    openai_api_key = open_api_key,
    model='gpt-3.5-turbo-instruct',
    temperature=0,)
retriever = SelfQueryRetriever.from_llm(
    llm,
    vectordb,
    document_content_description,
    metadata_field_info,
    verbose=True
)

In [66]:
question = "what did they say about supply chain in the chapter Impacts of technology disruption?"
docs = retriever.get_relevant_documents(question)
for d in docs:
    print(d.metadata)

In [67]:
len(docs)

0

#### Contextual Compression Retriever and LLMChainExtractor

Often the RAG retrieves the entire set of docs, while only a section or a few lines from the retrieved docs are relevant to the context/question. Contextual Compression (using LLM) can help narrow down the response to only the relevant text.

In [68]:
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import LLMChainExtractor

def pretty_print(docs):
  print("\n\n".join([x.page_content for x in docs]))

prompt = f"""
Hello LLM. I am using you as doc retriever, and a contextual compressor.

Pick as many relevant documents as possible, and as diverse as possible
""".strip()

### DOES NOT WORK WITH ARG prompt
# Error:
'''
/usr/local/lib/python3.11/dist-packages/langchain_community/llms/openai.py in _generate(self, prompts, stop, run_manager, **kwargs)
    461                 )
    462             else:
--> 463                 response = completion_with_retry(
    464                     self, prompt=_prompts, run_manager=run_manager, **params
    465                 )

TypeError: langchain_community.llms.openai.completion_with_retry() got multiple values for keyword argument 'prompt'
'''

#llm = OpenAI(openai_api_key=open_api_key, temperature=0, prompt=prompt)
#print(llm.get_num_tokens_from_messages(messages=prompt))

llm = OpenAI(openai_api_key=open_api_key, temperature=0)

print(client.get_api_list)
#print(llm.get_api_list)

compressor = LLMChainExtractor.from_llm(llm)
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=vectordb.as_retriever()
)

# Asking the same question
'''
"what did they say about supply chain in the chapter Impacts of technology disruption?"
'''
compressed_docs = compression_retriever.get_relevant_documents(
    query = question,
    #prompt = prompt
    )

pretty_print(compressed_docs)


<bound method SyncAPIClient.get_api_list of <openai.OpenAI object at 0x7a512655c210>>
- Dramatic technological and digital developments, 
such as greater computing power and lower overall 
costs, have impacted the traditional supply chain in 
several key ways, including a reduction in transac-
tion costs and increase in innovation related to the 
production process itself.
- Reducing transaction costs
- The increase in power and efficiency of technologies 
has manifested itself in greatly reduced transaction 
costs for business operations both internally and ex-
ternally.8 No longer does it have to be prohibitively 
expensive or time intensive to gain insight into each 
minute step of operations, or to deeply understand 
customer or supplier demand patterns.

The new interconnections between processes and subpro-
cesses have transformed supply chains into efficient 
and predictive networks. When the cost of trans-
actions falls, the ability to transact with more and 
different partners

# Chat Bot
Let's check the relevant docs using Vector DB similarity search...

In [72]:
testQ = "What are the topics covered in the class"
testQ_docs = vectordb.similarity_search(testQ, k=3)
print("Searched docs: ", len(testQ_docs))
print("\n\n-----\n".join([x.page_content for x in testQ_docs]))

Searched docs:  3
Use Cases
• Defining key result areas of a specific role in the organisation e.g. procurement 
manager of a manufacturing company.
• Preparing a question bank for interview for specific roles in an organisation along with 
model answers e.g. question bank for interviewing for a welder position.
• Getting a suggestion on feasibility of a new business idea.
• Creating a marketing collateral that will communicate the benefit of a product of a 
company over other competing products.
• Listing a few investing principles for beginners investing in stock market.

-----
Tool Use
•Analysis
• Code Execution
• Wolfram Alpha
•Research
• Search Engine
• Web Browsing
• Wikipedia
•Productivity
• Email
• Calendar
• Cloud Storage
•Images
• Image Generation
• Image  captioning
• Object Detection
Reference: Andrew Ng’s Talk at Sequoia Capital

-----
Prompts
You are a friendly and helpful tutor. Your job is to explain a concept to the user in a clear and straightforward way, 
give the us

### Create OpenAI ChatBot

In [75]:
from langchain.chat_models import ChatOpenAI
from langchain.schema import (
    AIMessage,
    HumanMessage,
    SystemMessage
)

# create OpenAI chat model
chat_llm = ChatOpenAI(
    openai_api_key=open_api_key,
    model_name='gpt-3.5-turbo',
    temperature=0)



/tmp/ipython-input-2972327536.py:11: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import ChatOpenAI``.
  chat_llm = ChatOpenAI(


### RetrievalQA

In [81]:
# Test QA retrieval using "RetrievalQA"
# QA ==> Question Answering
from langchain.chains import LLMChain, RetrievalQA, ConversationalRetrievalChain

qa_chain = RetrievalQA.from_chain_type(
    llm=chat_llm,
    #chain_type="stuff",
    retriever=vectordb.as_retriever()
)

print(qa_chain({"query": testQ})['result'])
print(qa_chain(
    {"query": 'dont ask me. just answer'})['result'])

Hello! Before we dive into the topics covered in the class, I'd like to ask you a couple of questions to better understand your learning level and interests. 

Question 1: Could you please tell me about your learning level? Are you in high school, college, or a professional setting?
Question 2: What specific topic or concept would you like to know more about?
Question 3: Why does this topic interest you?
Question 4: What do you already know about the topic?

Feel free to answer those questions, and then I'll be able to provide you with a tailored explanation of the topics covered in the class.
I'm here to help you understand a concept clearly and straightforwardly. Let's talk about the concept of "gravity." Gravity is the force that pulls objects towards each other. Imagine gravity as a magnet that pulls things together. For example, when you drop a ball, gravity pulls it down towards the ground. The reason why things fall to the ground is because of gravity.

Now, let's check your und

### ConversationalRetrievalChain
To be done!

In [92]:
# Use conversational retrieval chain - TBD!
# TODO!!
'''
cr_chain = ConversationalRetrievalChain.from_llm(
    llm=chat_llm,
    retriever=vectordb.as_retriever(),
    # Need chat_history
    # chat_history=[]
    # Need memory - will see later
    # memory=memory,
    # verbose=True
)

print(cr_chain({"question": testQ})['answer'])
'''

'\ncr_chain = ConversationalRetrievalChain.from_llm(\n    llm=chat_llm,\n    retriever=vectordb.as_retriever(),\n    # Need chat_history\n    # chat_history=[]\n    # Need memory - will see later\n    # memory=memory,\n    # verbose=True\n)\n\nprint(cr_chain({"question": testQ})[\'answer\'])\n'

### RetrievalQA With Prompt Template

In [ ]:
from langchain.prompts import PromptTemplate
# Build prompt
prompt_template = """
Use the following context to answer the question asked by the
user to the point. And, do not cross question the user or punt it back.
Be precise and to the point.

Context: {context}

Question: {question}

Helpful Answer: """
PROMPT = PromptTemplate(
    template=prompt_template, input_variables=["context", "question"])



#### Map Reduce

#### Refine

#### Re-Rank